In [1]:
%run ../run_config_project.py
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data


You are working with      IGVF BlueSTARR
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references



In [3]:
import numpy  as np
import pandas as pd
import pickle
import os

In [11]:
import sklearn

In [6]:
!ls /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_full
batches_pilot
JASPAR2024_CORE_vertebrates_non-redundant.lods.npz
JASPAR2024_CORE_vertebrates_non-redundant.lods.pkl
JASPAR2024_CORE_vertebrates_non-redundant.pmap.npz
JASPAR2024_CORE_vertebrates_non-redundant.pmap.pkl
motifdelta_pilot_jaspar2024
motifdelta_pilot_jvierstra_v2.0beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_nonredundant_jvierstra_v2.0beta.lods.pkl
motif_nonredundant_jvierstra_v2.0beta.pmap.pkl
tmp
variant_closed_gof_bluestarr_flank35_obs.fa
variant_closed_gof_bluestarr_flank35_ref.fa
variant_closed_gof_bluestarr_flank35_unobs.fa
variant_closed_gof_bluestarr_flankL35R70_obs.fa
variant_closed_gof_bluestarr_flankL35R70_ref.fa
variant_closed_gof_bluestarr_flankL35R70_unobs.fa
variant_closed_gof_bluestarr.tsv.gz
variant_closed_gof_bluestarr_withseq_flank35.tsv.gz
variant_closed_gof_bluestarr_withseq_flankL35R70.tsv.gz


In [4]:
txt_fdiry = f"{FD_RES}/analysis_variant_motif_richard"

print(txt_fdiry)

/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard


In [9]:
txt_fdiry = f"{FD_RES}/analysis_variant_motif_richard"
txt_fname = "motif_nonredundant_jvierstra_v2.0beta.lods.pkl"
txt_fpath = f"{txt_fdiry}/{txt_fname}"

with open(txt_fpath, "rb") as f:
    obj = pickle.load(f)

dct_motif_lods = obj["lods"]

In [10]:
list(dct_motif_lods.keys())[:6]

['AC0001:DLX/LHX:Homeodomain AC0001:DLX/LHX:Homeodomain',
 'AC0002:EMX/PAX:Homeodomain AC0002:EMX/PAX:Homeodomain',
 'AC0003:HOXB/BARHL:Homeodomain AC0003:HOXB/BARHL:Homeodomain',
 'AC0004:PAX/LHX:Homeodomain AC0004:PAX/LHX:Homeodomain',
 'AC0005:POU6F/NKX:Homeodomain AC0005:POU6F/NKX:Homeodomain',
 'AC0006:POU3F/POU1F:Homeodomain,POU AC0006:POU3F/POU1F:Homeodomain,POU']

In [4]:
import pandas as pd


fp = f"{FD_RES}/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_event.tsv"
df = pd.read_csv(fp, sep="\t")

# 1) make sure we only have "Gain" and "Loss"
df["Event_Type"].value_counts()

# 2) confirm at most one row per (Variant, Motif, Event_Type)
dup = df.duplicated(subset=["Variant_ID", "Motif_Name", "Event_Type"])
dup.sum()

0

In [5]:
# Load summaries
fp_var = f"{FD_RES}/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_summary_variant.tsv"
fp_mot = f"{FD_RES}/analysis_variant_motif_richard/motifdelta_pilot/variant_closed_gof_bluestarr_flankL35R70_pilot_chunk001_summary_motif.tsv"

dat_var = pd.read_csv(fp_var, sep="\t")
dat_mot = pd.read_csv(fp_mot, sep="\t")

# Variant-level: recompute from events
var_counts = (
    df.pivot_table(
        index="Variant_ID",
        columns="Event_Type",
        values="Motif_Name",
        aggfunc="nunique",  # # motifs per variant per type
    )
    .fillna(0)
    .rename(columns={"Gain": "Count_Gain", "Loss": "Count_Loss"})
    .reset_index()
)

# Compare to summary_variant (should match)
dat_var_merged = dat_var.merge(var_counts, on="Variant_ID", suffixes=("_summary", "_from_events"))
(dat_var_merged["Count_Gain_summary"] != dat_var_merged["Count_Gain_from_events"]).sum(), \
(dat_var_merged["Count_Loss_summary"] != dat_var_merged["Count_Loss_from_events"]).sum()

(0, 0)

In [6]:
mot_counts = (
    df.pivot_table(
        index="Motif_Name",
        columns="Event_Type",
        values="Variant_ID",
        aggfunc="nunique",  # # variants per motif per type
    )
    .fillna(0)
    .rename(columns={"Gain": "Count_Gain", "Loss": "Count_Loss"})
    .reset_index()
)

dat_mot_merged = dat_mot.merge(mot_counts, on="Motif_Name", suffixes=("_summary","_from_events"))
(
    (dat_mot_merged["Count_Gain_summary"] != dat_mot_merged["Count_Gain_from_events"]).sum(),
    (dat_mot_merged["Count_Loss_summary"] != dat_mot_merged["Count_Loss_from_events"]).sum()
)

(0, 0)